In [ ]:
!pip install transformers datasets librosa torchaudio accelerate matplotlib

In [ ]:
import torch
import numpy as np
from transformers import TrainingArguments, Trainer
import matplotlib.pyplot as plt
import seaborn as sns

from transformers import AutoFeatureExtractor, AutoModelForAudioClassification

from datasets import load_dataset, Audio, DatasetDict

###Φόρτωση dataset και speaker-independent διαχωρισμός

In [ ]:
def load_tess_speaker_independent():
    dataset = load_dataset("AbstractTTS/TESS", split="train")
    dataset = dataset.cast_column("audio", Audio(decode=False))

    def get_speaker(example):
        filename = example["audio"]["path"]
        if "OAF" in filename:
            return "OAF"
        elif "YAF" in filename:
            return "YAF"
        return "unknown"

    train_dataset = dataset.filter(lambda x: get_speaker(x) == "OAF")
    test_dataset = dataset.filter(lambda x: get_speaker(x) == "YAF")

    train_dataset = train_dataset.cast_column("audio", Audio(sampling_rate=16_000))
    test_dataset = test_dataset.cast_column("audio", Audio(sampling_rate=16_000))

    return DatasetDict({"train": train_dataset, "test": test_dataset})

In [ ]:
dataset = load_tess_speaker_independent()

In [ ]:
labels = sorted(set(dataset["train"]["emotion"]))
label2id = {name: i for i, name in enumerate(labels)}
id2label = {i: name for i, name in enumerate(labels)}
num_labels = len(labels)
print("Κλάσεις:", labels)

In [ ]:
from transformers import WhisperForAudioClassification, AutoFeatureExtractor

checkpoint = "openai/whisper-base"

feature_extractor = AutoFeatureExtractor.from_pretrained(checkpoint)
model = WhisperForAudioClassification.from_pretrained(
    checkpoint, num_labels=num_labels, label2id=label2id, id2label=id2label,
    ignore_mismatched_sizes=True,
).to("cuda")

###Προετοιμασία dataset

In [ ]:
#προετοιμασία dataset
def prepare_dataset(batch):
    audio = batch["audio"]
    batch["input_features"] = feature_extractor(
        audio["array"], sampling_rate=audio["sampling_rate"]
    ).input_features[0]
    batch["labels"] = label2id[batch["emotion"]]
    return batch

encoded_dataset = dataset.map(prepare_dataset, remove_columns=dataset["train"].column_names, load_from_cache_file=False)

###Data Collator

In [ ]:
import torch

class DataCollatorWhisperClassification:
    def __init__(self, feature_extractor):
        self.feature_extractor = feature_extractor

    def __call__(self, features):
        input_features = torch.stack([torch.tensor(f["input_features"]) for f in features])
        labels = torch.tensor([f["labels"] for f in features], dtype=torch.long)
        return {"input_features": input_features, "labels": labels}

data_collator = DataCollatorWhisperClassification(feature_extractor)

### Ορισμός μετρικών αξιολόγησης

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,

    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    cohen_kappa_score,
    roc_auc_score,
    log_loss,
    confusion_matrix,

    average_precision_score
)

from sklearn.preprocessing import label_binarize
from scipy.special import softmax

In [ ]:
#Λίστα για την αποθήκευση των confusion matrices
all_confusion_matrices = []

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    probs = softmax(logits, axis=1)

    num_classes = probs.shape[1]
    labels_bin = label_binarize(labels, classes=list(range(num_classes)))

    # ROC-AUC (macro-averaged, one-vs-rest)
    roc_auc = roc_auc_score(
        labels_bin, probs, multi_class="ovr", average="macro"
    )

    # PR-AUC (macro-averaged average precision)
    pr_auc = average_precision_score(
        labels_bin, probs, average="macro"
    )

    accuracy = accuracy_score(labels, predictions)
    balanced_accuracy = balanced_accuracy_score(labels, predictions)
    precision = precision_score(labels, predictions, average="weighted", zero_division=0)
    recall = recall_score(labels, predictions, average="weighted")
    f1 = f1_score(labels, predictions, average="weighted")
    mcc = matthews_corrcoef(labels, predictions)
    k_score = cohen_kappa_score(labels, predictions)
    logloss = log_loss(labels, probs)

    cm = confusion_matrix(labels, predictions)
    all_confusion_matrices.append(cm)

    return {
        "accuracy": accuracy,
        "balanced_accuracy": balanced_accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "mcc" : mcc,
        "k_score" : k_score,
        "log_loss": logloss,
        "roc_auc": roc_auc,
        "pr_auc": pr_auc
    }

### Ορισμός παραμέτρων εκπαίδευσης

In [ ]:
# Δυναμικός υπολογισμός warmup_steps (10% των συνολικών training steps)
per_device_train_batch_size = 4
gradient_accumulation_steps = 4
num_train_epochs = 10

effective_batch_size = per_device_train_batch_size * gradient_accumulation_steps
steps_per_epoch = len(encoded_dataset["train"]) // effective_batch_size
total_steps = steps_per_epoch * num_train_epochs
warmup_steps = max(1, int(0.1 * total_steps))


training_args = TrainingArguments(
    output_dir="./whisper-finetuned-TESS",
    per_device_train_batch_size=per_device_train_batch_size,
    gradient_accumulation_steps=gradient_accumulation_steps,
    per_device_eval_batch_size=8,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    num_train_epochs=num_train_epochs,
    fp16=False,
    learning_rate=3e-5,
    warmup_steps=warmup_steps,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    push_to_hub=False,
)

In [ ]:
#Καθαρισμός των confusion matrix
all_confusion_matrices.clear()

### Δημιουργία και Εκπαίδευση του Trainer

In [ ]:
#Κλήση του trainer και εισαγωγή των παραμέτρων εκπαίδευσης
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=encoded_dataset["train"],
    eval_dataset=encoded_dataset["test"],
    compute_metrics=compute_metrics,
    data_collator=data_collator,
)

print("Εκκίνηση εκπαίδευσης...")
trainer.train()
print("Εκπαίδευση ολοκληρώθηκε!")

In [ ]:
#Εκτύπωση των τελικών αποτελεσμάτων της εκπαίδευσης
eval_results = trainer.evaluate()
print(f"Αποτελέσματα αξιολόγησης: {eval_results}")

### Οπτικοποίηση αποτελεσμάτων (Confusion Matrices)

In [ ]:
#Αποθήκευση των ετικετών για τους άξονες των matrices
labels = [
    model.config.id2label[i]
    for i in sorted(model.config.id2label.keys())
]

#Loop στην λίστα με τα confusion matrices
for epoch, cm in enumerate(all_confusion_matrices):
  #Δημιοργία γραφήματος
  plt.figure(figsize=(6,5))

  #Δηνιουργία Heatmap
  sns.heatmap(
      cm,
      annot=True,
      fmt="d",
      xticklabels=labels,
      yticklabels=labels
  )

  plt.title(f"Confusion Matrix - Epoch {epoch+1}")
  plt.xlabel("Predicted")
  plt.ylabel("True")
  plt.show()

### Αποθήκευση εκπαιδευμένου μοντέλου στο HuggingFace Hub

In [ ]:
from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get('HF_TOKEN'))

In [ ]:
repo_id = "VasilisAsim/whisper-finetuned-TESS"

#Αποθήκευση του μοντέλου στο HuggingFace
model.push_to_hub(repo_id)

#Αποθήκευση του Feature Extractor
feature_extractor.push_to_hub(repo_id)

print(f"Το μοντέλο και ο επεξεργαστής αποθηκεύτηκαν στο Hugging Face Hub με ID: {repo_id}")